In [56]:
# Define some variables here
_NYT_CONNECTION_FILE = "/Users/yt/Desktop/nyt-connections-rag/data/ConnectionsFinalDataset.json"
#_NYT_CONNECTION_FILE = '/home/vgupta8/cs429/nyt-connections-rag/data/ConnectionsFinalDataset.json'
#_DIFFICULTY = range(3, 5)  # 1=Yellow, 2=Green, 3=Blue, 4=Purple; use range(1,5) for all
_GAME_DATE = ""
_NUMBER_OF_CONNECTIONS = 5
#_GITHUB_TOKEN = "GITHUB_TOKEN"
#_URL =https://models.inference.ai.azure.com
_GitHUB_TOKEN = "OPENROUTER_API_KEY"
_URL ="https://openrouter.ai/api/v1"


In [2]:
# Helper functions for wordnet, evaluating connections, and other related tasks

from nltk.corpus import wordnet as wn

def get_word_synsets(words):
    """Return a dict mapping each word to its unique lemma names from WordNet."""
    result = {}
    for word in words:
        synsets = wn.synsets(word.lower().replace(" ", "_"))
        result[word] = list(dict.fromkeys(
            l.name().replace("_", " ") for s in synsets for l in s.lemmas()
        ))
    return result

## Evaluation metrics for predicted groups vs actual answers

def game_completion_rate(all_predicted, all_actual):
    """All 4 groups exactly correct → puzzle is 'completed'."""
    if not all_predicted:
        return 0.0
    completed = 0
    for predicted, actual in zip(all_predicted, all_actual):
        actual_sets = [set(a["words"]) for a in actual]
        pred_sets   = [set(p["words"]) for p in predicted]
        if all(ps in actual_sets for ps in pred_sets):
            completed += 1
    return completed / len(all_predicted)


def overall_accuracy(all_predicted, all_actual):
    """Proportion of predicted groups that exactly match a ground-truth group."""
    total, correct = 0, 0
    for predicted, actual in zip(all_predicted, all_actual):
        actual_sets = [set(a["words"]) for a in actual]
        for pred in predicted:
            total += 1
            if set(pred["words"]) in actual_sets:
                correct += 1
    return correct / total if total else 0.0


def partial_accuracy_rate(all_predicted, all_actual):
    """
    Proportion of predicted groups whose best overlap with any ground-truth group
    is exactly 3 or exactly 2 words (i.e. near-correct but not exact).
    Returns dict: {3: rate_for_3_overlap, 2: rate_for_2_overlap}
    """
    total = 0
    counts = {2: 0, 3: 0}
    for predicted, actual in zip(all_predicted, all_actual):
        actual_sets = [set(a["words"]) for a in actual]
        for pred in predicted:
            pred_set = set(pred["words"])
            best_overlap = max(len(pred_set & a_set) for a_set in actual_sets)
            total += 1
            if best_overlap in counts:
                counts[best_overlap] += 1
    return {k: v / total if total else 0.0 for k, v in counts.items()}



## Evaluation helpers — run once, then call evaluate_run() in the cells below

from IPython.display import display, HTML

def _badge(text, bg, fg="#fff"):
    return (f'<span style="background:{bg};color:{fg};padding:2px 8px;'
            f'border-radius:4px;font-size:0.85em;font-weight:bold">{text}</span>')

def render_puzzle(idx, total, contest, raw_reply, predicted, actual, scores):
    correct_groups = scores["correct_groups"]
    completed      = scores["completed"]
    group_acc      = scores["group_acc"]
    actual_sets    = [set(a["words"]) for a in actual]

    status_badge = (_badge("✓ SOLVED", "#2e7d32") if completed
                    else _badge(f"✗ {correct_groups}/4", "#c62828"))

    html = []
    html.append('<div style="border:1px solid #555;border-radius:8px;padding:16px;'
                'margin:12px 0;font-family:monospace">')

    # Header
    html.append('<div style="display:flex;align-items:center;gap:12px;margin-bottom:12px">')
    html.append(f'<b style="font-size:1.1em">[{idx}/{total}] {contest}</b>')
    html.append(status_badge)
    html.append(f'<span style="color:#888;font-size:0.85em">'
                f'acc {group_acc:.0%} | partial-3 {scores["partial_3"]:.0%} | partial-2 {scores["partial_2"]:.0%}'
                f'</span>')
    html.append('</div>')

    # Raw reply (collapsible)
    escaped = raw_reply.replace("&","&amp;").replace("<","&lt;").replace(">","&gt;")
    html.append('<details style="margin-bottom:12px">')
    html.append('<summary style="cursor:pointer;color:#90caf9">▶ Raw model reply</summary>')
    html.append(f'<pre style="background:#1e1e1e;color:#d4d4d4;padding:10px;border-radius:6px;'
                f'overflow-x:auto;white-space:pre-wrap;font-size:0.85em;margin-top:6px">{escaped}</pre>')
    html.append('</details>')

    # Side-by-side table
    html.append('<table style="width:100%;border-collapse:collapse;font-size:0.9em">')
    html.append('<tr>'
                '<th style="text-align:left;padding:4px 8px;border-bottom:1px solid #444;width:50%;color:#aaa">Predicted</th>'
                '<th style="text-align:left;padding:4px 8px;border-bottom:1px solid #444;width:50%;color:#aaa">Actual</th>'
                '</tr>')

    for row in range(max(len(predicted), len(actual))):
        html.append('<tr style="vertical-align:top">')

        if row < len(predicted):
            pred = predicted[row]
            ps = set(pred["words"])
            is_correct = ps in actual_sets
            bg = "#1b3a1b" if is_correct else "#3a1a1a"
            icon_color = "#66bb6a" if is_correct else "#ef5350"
            icon = "✓" if is_correct else "✗"
            label = pred.get("answerDescription", "?")
            if is_correct:
                words_display = [f'<span style="color:#66bb6a">{w}</span>' for w in pred["words"]]
                note = ""
            else:
                best_a = max(actual, key=lambda a: len(ps & set(a["words"])))
                best_set = set(best_a["words"])
                words_display = [
                    f'<span style="color:{"#66bb6a" if w in best_set else "#ef5350"}">{w}</span>'
                    for w in pred["words"]
                ]
                overlap = len(ps & best_set)
                note = (f'<div style="color:#888;font-size:0.8em;margin-top:2px">'
                        f'best match: <i>{best_a["answerDescription"]}</i> ({overlap}/4)</div>')
            html.append(
                f'<td style="padding:5px 8px;background:{bg};border-radius:4px">'
                f'<span style="color:{icon_color}">{icon}</span> '
                f'<b style="color:#ccc">{label}</b><br>'
                f'{", ".join(words_display)}{note}</td>'
            )
        else:
            html.append('<td></td>')

        if row < len(actual):
            a = actual[row]
            diff_colors = {"Yellow": "#f9a825", "Green": "#2e7d32",
                           "Blue": "#1565c0", "Purple": "#6a1b9a"}
            cat_color = diff_colors.get(a.get("difficulty", ""), "#555")
            html.append(
                f'<td style="padding:5px 8px;background:#1e1e2e;border-radius:4px">'
                f'<b style="color:{cat_color}">{a["answerDescription"]}</b><br>'
                f'<span style="color:#ccc">{", ".join(a["words"])}</span></td>'
            )
        else:
            html.append('<td></td>')

        html.append('</tr>')

    html.append('</table></div>')
    return "".join(html)


def evaluate_run( raw_replies, all_actual):
    """Evaluate a list of raw model replies against ground-truth answers. Returns per_puzzle_scores."""
    display(HTML(
        f'<h2 style="font-family:monospace;border-bottom:2px solid #555;padding-bottom:6px">'
    ))

    predicted_all  = []
    per_puzzle_scores = []

    for i, (raw_reply, actual) in enumerate(zip(raw_replies, all_actual)):
        try:
            s = raw_reply.find('[')
            e = raw_reply.rfind(']') + 1
            predicted = json.loads(raw_reply[s:e])
        except Exception as ex:
            print(f"[{i+1}] Failed to parse JSON: {ex}")
            predicted = []
        predicted_all.append(predicted)

        actual_sets = [set(a["words"]) for a in actual]
        pred_sets   = [set(p["words"]) for p in predicted]

        correct_groups = sum(1 for ps in pred_sets if ps in actual_sets)
        completed      = int(all(ps in actual_sets for ps in pred_sets) and len(pred_sets) == 4)
        partial_3 = sum(
            1 for ps in pred_sets
            if max((len(ps & a) for a in actual_sets), default=0) >= 3
        ) / max(len(pred_sets), 1)
        partial_2 = sum(
            1 for ps in pred_sets
            if max((len(ps & a) for a in actual_sets), default=0) >= 2
        ) / max(len(pred_sets), 1)
        group_acc = correct_groups / max(len(pred_sets), 1)

        scores = dict(correct_groups=correct_groups, completed=completed,
                      group_acc=group_acc, partial_3=partial_3, partial_2=partial_2)
        per_puzzle_scores.append({"contest": pool[i]["contest"], **scores})
        display(HTML(render_puzzle(i + 1, len(pool), pool[i]["contest"],
                                   raw_reply, predicted, actual, scores)))

    # Summary table
    n = len(per_puzzle_scores)
    avg_gcr = sum(s["completed"]  for s in per_puzzle_scores) / n
    avg_acc = sum(s["group_acc"]  for s in per_puzzle_scores) / n
    avg_p3  = sum(s["partial_3"]  for s in per_puzzle_scores) / n
    avg_p2  = sum(s["partial_2"]  for s in per_puzzle_scores) / n

    header = (
        '<tr style="color:#aaa;border-bottom:1px solid #555">'
        '<th style="padding:4px 12px;text-align:left">Contest</th>'
        '<th style="padding:4px 12px">Solved</th>'
        '<th style="padding:4px 12px">Group Acc</th>'
        '<th style="padding:4px 12px">Partial-3</th>'
        '<th style="padding:4px 12px">Partial-2</th></tr>'
    )
    rows = "".join(
        f'<tr><td style="padding:4px 12px">{s["contest"]}</td>'
        f'<td style="padding:4px 12px;text-align:center">{"✓" if s["completed"] else "✗"}</td>'
        f'<td style="padding:4px 12px;text-align:center">{s["group_acc"]:.0%}</td>'
        f'<td style="padding:4px 12px;text-align:center">{s["partial_3"]:.0%}</td>'
        f'<td style="padding:4px 12px;text-align:center">{s["partial_2"]:.0%}</td></tr>'
        for s in per_puzzle_scores
    )
    avg_row = (
        f'<tr style="border-top:2px solid #666;font-weight:bold">'
        f'<td style="padding:6px 12px">Average ({n})</td>'
        f'<td style="padding:6px 12px;text-align:center">{avg_gcr:.1%}</td>'
        f'<td style="padding:6px 12px;text-align:center">{avg_acc:.1%}</td>'
        f'<td style="padding:6px 12px;text-align:center">{avg_p3:.1%}</td>'
        f'<td style="padding:6px 12px;text-align:center">{avg_p2:.1%}</td></tr>'
    )
    display(HTML(
        f'<h3 style="font-family:monospace;margin-top:24px">Summary — {n} puzzle{"s" if n!=1 else ""}</h3>'
        f'<table style="border-collapse:collapse;font-family:monospace;font-size:0.9em">'
        f'{header}{rows}{avg_row}</table>'
    ))

    return per_puzzle_scores, dict(gcr=avg_gcr, acc=avg_acc, p3=avg_p3, p2=avg_p2)



In [3]:
# Function for GPT to call WordNet

def analyze_wordnet_similarity(words):
    """Return pairwise WordNet similarity scores for the provided words."""
    similarities = {}
    for i, word1 in enumerate(words):
        for j, word2 in enumerate(words):
            if i < j:
                try:
                    syn1 = wn.synsets(word1.lower())[0] if wn.synsets(word1.lower()) else None
                    syn2 = wn.synsets(word2.lower())[0] if wn.synsets(word2.lower()) else None
                    best_score = 0
                    if syn1 and syn2:
                        candidates = []
                        try:
                            candidates.append(syn1.path_similarity(syn2))
                        except Exception:
                            pass

                        for ic in (globals().get('brown_ic'), globals().get('semcor_ic')):
                            if ic is not None:
                                try:
                                    candidates.append(syn1.res_similarity(syn2, ic))
                                except Exception:
                                    pass
                                try:
                                    candidates.append(syn1.lin_similarity(syn2, ic))
                                except Exception:
                                    pass
                                try:
                                    candidates.append(syn1.jcn_similarity(syn2, ic))
                                except Exception:
                                    pass

                        try:
                            candidates.append(syn1.wup_similarity(syn2))
                        except Exception:
                            pass

                        valid_scores = [score for score in candidates if score is not None]
                        if valid_scores:
                            best_score = max(valid_scores)
                    similarities[f"{word1}-{word2}"] = round(best_score, 3)
                except:
                    similarities[f"{word1}-{word2}"] = 0
    return similarities


def analyze_wordnet_hypernyms(words):
    """Return the top hypernyms for each word."""
    hypernyms = {}
    for word in words:
        try:
            syn = wn.synsets(word.lower())[0] if wn.synsets(word.lower()) else None
            if syn:
                hypernyms[word] = [h.name().split('.')[0] for h in syn.hypernyms()[:3]]
            else:
                hypernyms[word] = []
        except:
            hypernyms[word] = []
    return hypernyms


def analyze_wordnet_definitions(words):
    """Return the primary WordNet definition for each word."""
    definitions = {}
    for word in words:
        try:
            syn = wn.synsets(word.lower())[0] if wn.synsets(word.lower()) else None
            if syn:
                definitions[word] = syn.definition()
            else:
                definitions[word] = "No definition found"
        except:
            definitions[word] = "Error getting definition"
    return definitions


def analyze_wordnet_relationships(words):
    """
    Analyze relationships between words using WordNet.

    Args:
        words: List of words to analyze

    Returns:
        Dictionary with all available analysis results.
    """
    return {
        "similarity": analyze_wordnet_similarity(words),
        "hypernyms": analyze_wordnet_hypernyms(words),
        "definitions": analyze_wordnet_definitions(words),
    }

# NYT Connections game data
game_data = {
    "words": [
        "LASER", "PLUCK", "THREAD", "WAX",
        "COIL", "SPOOL", "WIND", "WRAP",
        "HONEYCOMB", "ORGANISM", "SOLAR PANEL", "SPREADSHEET",
        "BALL", "MOVIE", "SCHOOL", "VITAMIN"
    ]
}

In [38]:
import ipywidgets as widgets
from IPython.display import display, Markdown

import os
import json
from dotenv import load_dotenv
from openai import OpenAI

load_dotenv()

client = OpenAI(
     base_url= _URL,
     api_key=os.getenv(_GitHUB_TOKEN)
   # base_url="https://models.inference.ai.azure.com",
   # api_key=os.getenv("GITHUB_TOKEN")
)



# Function definitions for GPT
tools = [
    {
        "type": "function",
        "function": {
            "name": "analyze_wordnet_relationships",
            "description": "Analyze semantic relationships between words using WordNet",
            "parameters": {
                "type": "object",
                "properties": {
                    "words": {
                        "type": "array",
                        "items": {"type": "string"},
                        "description": "List of words to analyze"
                    }
                },
                "required": ["words"]
            }
        }
    }
]

def ask_gpt(messages, prompt, use_tools):
    # Use a fresh local conversation each call, keeping only the system prompt
    local_messages = [messages[0], {"role": "user", "content": prompt}]

    kwargs = {}
    if use_tools:
        kwargs["tools"] = tools
        kwargs["tool_choice"] = "auto"

    completion = client.chat.completions.create(
        model="gpt-4o-mini",
        messages=local_messages,
        **kwargs
    )

    if completion.choices and len(completion.choices) > 0:
        message = completion.choices[0].message
        tool_calls = getattr(message, "tool_calls", None)
        #print("Tools:", tool_calls)
        if tool_calls:
            # Append the assistant's reply (with tool_calls) to the conversation
            local_messages.append(message)

            for tool_call in tool_calls:
                tool_name = tool_call.function.name
                tool_args = json.loads(tool_call.function.arguments)

                if tool_name == "analyze_wordnet_relationships":
                    tool_result = analyze_wordnet_relationships(**tool_args)
                else:
                    tool_result = {"error": f"Unknown tool: {tool_name}"}

                # tool_call_id is required to match this result to the right call
                local_messages.append({
                    "role": "tool",
                    "tool_call_id": tool_call.id,
                    "content": json.dumps(tool_result)
                })
            print("Local conversation with tool calls and results:",local_messages)
            followup = client.chat.completions.create(
                model="gpt-4o-mini",
                messages=local_messages
            )
            if followup.choices:
                return followup.choices[0].message.content

        return getattr(message, "content", None)
    return str(completion)




# Randomly select games based on diffculty

### Dataset 1 -- filter with diffculty

In [57]:
# Select Words:

import json
import random

with open(_NYT_CONNECTION_FILE) as f:
    dataset = json.load(f)

##Filters
GAME_DATE             = _GAME_DATE              # Set to None or "" to skip date filter (format: "YYYY/MM/DD")
SAMPLES_PER_BAND      = _NUMBER_OF_CONNECTIONS  # Number of puzzles to sample per difficulty band

difficulty_bands = [(1, 2), (2, 3), (3, 4), (4, 5)]

pool = []
for lo, hi in difficulty_bands:
    band = [p for p in dataset if p.get("difficulty") is not None and lo <= p["difficulty"] < hi]
    if GAME_DATE:
        band = [p for p in band if p["date"] == GAME_DATE]
    sampled = random.sample(band, min(SAMPLES_PER_BAND, len(band)))
    pool.extend(sampled)
    print(f"  difficulty {lo}-{hi}: {len(band)} available, selected {len(sampled)}")

if not pool:
    raise ValueError(f"No puzzles found.")

print(f"\nTotal selected: {len(pool)} puzzle(s)")


  difficulty 1-2: 12 available, selected 5
  difficulty 2-3: 99 available, selected 5
  difficulty 3-4: 100 available, selected 5
  difficulty 4-5: 16 available, selected 5

Total selected: 20 puzzle(s)


# RAG Prep - Only Run Once

In [6]:
import os
from dotenv import load_dotenv
from openai import OpenAI
import nltk
import faiss
import numpy as np
import pickle
from sentence_transformers import SentenceTransformer

load_dotenv()




True

In [7]:
from nltk.corpus import wordnet as wn

def build_embedding_text(synset):
    definition = synset.definition()
    lemmas = ", ".join([str(lemma.name().replace("_", " ")) for lemma in synset.lemmas()])
    embedding = f"""The definition of {synset.name().split('.', 1)[0]} is {definition}. The words in this synset are: {lemmas}."""

    
    if len(synset.hypernyms()) != 0:
        hypernyms = ", ".join([str(hypernym.name()) for hypernym in synset.hypernyms()])
        embedding += f""" Hypernyms of this synset are: {hypernyms}."""
    if len(synset.examples()) != 0:
        examples = ", ".join([str(f"\"{example}\"") for example in synset.examples()])
        embedding += f""" Examples are: {examples}."""
    return embedding

all_embedding_sentences = []
metadata = []
for synset in wn.all_synsets():
    # if len(synset.lemmas()) >= 2:   # add if database building is slow since a lot of synsets seem to have only one word in them
    all_embedding_sentences.append(build_embedding_text(synset))
    metadata.append({
        "synset": synset.name(),
        "definition": synset.definition(),
        "lemmas": [lemma.name() for lemma in synset.lemmas()],
        "hypernyms":[hypernym.name() for hypernym in synset.hypernyms()],
        "pos": synset.pos()
    })

In [8]:
model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

embeddings = model.encode(all_embedding_sentences, show_progress_bar=True)

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

Batches:   0%|          | 0/3677 [00:00<?, ?it/s]

In [9]:
def normalize(vectors):
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / norms

normalized_embeddings = normalize(embeddings)
num_dimensions = embeddings.shape[1]
index = faiss.IndexFlatIP(num_dimensions)
index.add(normalized_embeddings) 

faiss.write_index(index, "wordnet.index")

with open("metadata.pkl", "wb") as f:
    pickle.dump(({"documents": all_embedding_sentences, "metadata": metadata}), f)


# RAG Perp - Start here after running above once

In [10]:
# Load FAISS index
index = faiss.read_index("wordnet.index")

# Load documents + metadata
with open("metadata.pkl", "rb") as f:
    data = pickle.load(f)

documents = data["documents"]
metadata = data["metadata"]

model = SentenceTransformer("sentence-transformers/all-MiniLM-L6-v2")

Loading weights:   0%|          | 0/103 [00:00<?, ?it/s]

In [49]:
def normalize(vectors):
    norms = np.linalg.norm(vectors, axis=1, keepdims=True)
    return vectors / norms

def search(query, k=5):
    query_vec = normalize(model.encode([query]))
    distances, indices = index.search(query_vec, k)


    results = []
    for i in indices[0]:
        results.append((documents[i], metadata[i]))


    return results

In [12]:
# function for Rag search
def RAG_search(words, k):
    """Return FAISS search results for each pair of words.""" 
    ##probably need a better explanation there, helppppp
    
    pair_results = {}
    for i, word1 in enumerate(words):
        for j, word2 in enumerate(words):
            if i < j:
                results = search(f"{word1} {word2}", k=k)
                pair_results[f"{word1}-{word2}"] = results
    return pair_results

# Baseline Model

In [58]:
#Baseline: no hints provied to GPT
raw_replies_without_hints = []
all_actual_without_hints = []
for i, puzzle in enumerate(pool):
    #message building for GPT prompt
    # System prompt for NYT Connections game
    messages = [
            {"role": "system", "content": f"""You are an expert at solving NYT Connections puzzles. 

        The puzzle has 16 words in 4 groups of 4. Each group shares a common theme. Analyze the relationships to find the groups.

        Current puzzle words: {", ".join(puzzle["words"])}

        Think step by step and use your knowledge to find the groups. 
        Return ONLY a JSON array with exactly 4 objects, each with:
        - "answerDescription": a short ALL-CAPS description of the category
        - "words": a list of exactly 4 words from the input

        Example format:
        [
          {{"answerDescription": "CATEGORY NAME", "words": ["WORD1", "WORD2", "WORD3", "WORD4"]}},
          ...
        ]
        """}
        ]  

    # Test GPT with Baseline (no hints)
    print("\nTesting GPT with Baseline (no hints) grouping...")
    group_prompt = (
        "Group the following 16 words into exactly 4 groups of 4 words each, " + ", ".join(puzzle["words"]) + ". "
        + "and return only the groups as lists: " )
    
    raw_reply = ask_gpt(messages, group_prompt, use_tools=False)
    raw_replies_without_hints.append(raw_reply)
    all_actual_without_hints.append(puzzle["answers"])
    print(f"  [{i+1}/{len(pool)}] done: {puzzle['contest']}")




Testing GPT with Baseline (no hints) grouping...
  [1/20] done: NYT Connections 188 - December 16th, 2023

Testing GPT with Baseline (no hints) grouping...
  [2/20] done: NYT Connections 272 - March 9th, 2024

Testing GPT with Baseline (no hints) grouping...
  [3/20] done: NYT Connections 296 - April 2nd, 2024

Testing GPT with Baseline (no hints) grouping...
  [4/20] done: NYT Connections 316 - April 22nd, 2024

Testing GPT with Baseline (no hints) grouping...
  [5/20] done: NYT Connections 264 - March 1st, 2024

Testing GPT with Baseline (no hints) grouping...
  [6/20] done: NYT Connections 300 - April 6th, 2024

Testing GPT with Baseline (no hints) grouping...
  [7/20] done: NYT Connections 213 - January 10th, 2024

Testing GPT with Baseline (no hints) grouping...
  [8/20] done: NYT Connections 155 - November 13th, 2023

Testing GPT with Baseline (no hints) grouping...
  [9/20] done: NYT Connections 185 - December 13th, 2023

Testing GPT with Baseline (no hints) grouping...
  [10/2

# Wordnet

In [59]:

#----------------With Wordnet------------------------------
raw_replies_with_hints = []
all_actual_with_hints = []
for i, puzzle in enumerate(pool):
    #message building for GPT prompt
    # System prompt for NYT Connections game
    messages = [
            {"role": "system", "content": f"""You are an expert at solving NYT Connections puzzles. You have access to WordNet to analyze semantic relationships between words.

        Use the analyze_wordnet_relationships function to:
        - Find similarities between words
        - Get hypernyms (broader categories) for words
        - Get definitions of words
        The function returns all supported analyses at once.

        The puzzle has 16 words in 4 groups of 4. Each group shares a common theme. Analyze the relationships to find the groups.

        Current puzzle words: {", ".join(puzzle["words"])}

        Think step by step and use WordNet data to support your reasoning.
        Return ONLY a JSON array with exactly 4 objects, each with:
        - "answerDescription": a short ALL-CAPS description of the category
        - "words": a list of exactly 4 words from the input

        Example format:
        [
          {{"answerDescription": "CATEGORY NAME", "words": ["WORD1", "WORD2", "WORD3", "WORD4"]}},
          ...
        ]
        """}
        ]
    
    
    
    #------------------------------------------
    # Test the WordNet integration
    result = analyze_wordnet_relationships(puzzle["words"])
    print("Similarities:", result["similarity"])
    print("Definitions:", result["definitions"])
    print("Hypernyms:", result["hypernyms"])

    # Test GPT with WordNet
    print("\nTesting GPT with WordNet grouping...")
    group_prompt = (
        "Group the following 16 words into exactly 4 groups of 4 words each, " + ", ".join(puzzle["words"]) + ". "
        + "and return only the groups as lists: " )
    
    raw_reply = ask_gpt(messages, group_prompt, use_tools=True)
    raw_replies_with_hints.append(raw_reply)
    all_actual_with_hints.append(puzzle["answers"])
    print(f"  [{i+1}/{len(pool)}] done: {puzzle['contest']}")


Similarities: {'HOE-PLOW': 0.889, 'HOE-RAKE': 0.421, 'HOE-SICKLE': 0.762, 'HOE-PLOT': 0.105, 'HOE-PLOY': 0.125, 'HOE-RUSE': 0.1, 'HOE-TRICK': 0.095, 'HOE-AMUSE': 0.125, 'HOE-DELIGHT': 0.125, 'HOE-PLEASE': 0.154, 'HOE-TICKLE': 0.105, 'HOE-BANG': 0.111, 'HOE-PLOP': 0.118, 'HOE-SPLASH': 0.118, 'HOE-THUD': 0.125, 'PLOW-RAKE': 0.421, 'PLOW-SICKLE': 0.762, 'PLOW-PLOT': 0.105, 'PLOW-PLOY': 0.125, 'PLOW-RUSE': 0.1, 'PLOW-TRICK': 0.095, 'PLOW-AMUSE': 0.125, 'PLOW-DELIGHT': 0.125, 'PLOW-PLEASE': 0.154, 'PLOW-TICKLE': 0.105, 'PLOW-BANG': 0.111, 'PLOW-PLOP': 0.118, 'PLOW-SPLASH': 0.118, 'PLOW-THUD': 0.125, 'RAKE-SICKLE': 0.364, 'RAKE-PLOT': 0.118, 'RAKE-PLOY': 0.143, 'RAKE-RUSE': 0.111, 'RAKE-TRICK': 0.105, 'RAKE-AMUSE': 0.133, 'RAKE-DELIGHT': 0.143, 'RAKE-PLEASE': 0.167, 'RAKE-TICKLE': 0.118, 'RAKE-BANG': 0.125, 'RAKE-PLOP': 0.133, 'RAKE-SPLASH': 0.133, 'RAKE-THUD': 0.143, 'SICKLE-PLOT': 0.091, 'SICKLE-PLOY': 0.105, 'SICKLE-RUSE': 0.087, 'SICKLE-TRICK': 0.083, 'SICKLE-AMUSE': 0.105, 'SICKLE-DELIG

# Rag

In [60]:
#----------------With RAG------------------------------
raw_replies_with_RAG = []
all_actual_with_RAG = []
for i, puzzle in enumerate(pool):
    #message building for GPT prompt
    # System prompt for NYT Connections game
    Semantic_info = RAG_search(puzzle["words"], k=2)
    messages = [
            {"role": "system", "content": f"""You are an expert at solving NYT Connections puzzles. You have access to WordNet to analyze semantic relationships between words.

        The puzzle has 16 words in 4 groups of 4. Each group shares a common theme. Analyze the relationships to find the groups.

        Current puzzle words: {", ".join(puzzle["words"])}
        
        You are given pairwise WordNet semantic information. Each key has the format WORD1-WORD2. Each value lists WordNet definitions, synsets, shared senses, synonyms, or hypernyms that may help explain a possible relationship.

        use below semantics infor to support the analysis:
        {json.dumps(Semantic_info, indent=2)}
        Important reasoning rules:
            1. Treat WordNet information as supporting evidence, not as the final answer.
            2. Some WordNet entries may be irrelevant because words can have multiple meanings.
            3. Look for clusters where several words share the same sense, synonym set, hypernym, or category.
            4. A valid group needs all 4 words to fit the same theme, not just one strong pair.
            5. NYT Connections often uses wordplay, idioms, slang, pop culture, fill-in-the-blank phrases, and category names that WordNet may not capture.
            6. Prefer groups where the relationship explains all 4 words naturally.
            7. Every input word must be used exactly once.
            8. No word may appear in more than one group.
            9. Each group must contain exactly 4 words.
       
        
        Return ONLY a JSON array with exactly 4 objects, each with:
        - "answerDescription": a short ALL-CAPS description of the category
        - "words": a list of exactly 4 words from the input

        Example format:
        [
          {{"answerDescription": "CATEGORY NAME", "words": ["WORD1", "WORD2", "WORD3", "WORD4"]}},
          ...
        ]    
        """}
        ]
    
    
    
    #------------------------------------------


    # Test GPT with RAG
    print("\nTesting GPT with RAG grouping...")
    group_prompt = (
        "Group the following 16 words into exactly 4 groups of 4 words each, " + ", ".join(puzzle["words"]) + ". "
        + "and return only the groups as lists: " )
    raw_reply = ask_gpt(messages, group_prompt, use_tools=False)
    raw_replies_with_RAG.append(raw_reply)
    all_actual_with_RAG.append(puzzle["answers"])
    print(f"  [{i+1}/{len(pool)}] done: {puzzle['contest']}")


Testing GPT with RAG grouping...
  [1/20] done: NYT Connections 188 - December 16th, 2023

Testing GPT with RAG grouping...
  [2/20] done: NYT Connections 272 - March 9th, 2024

Testing GPT with RAG grouping...
  [3/20] done: NYT Connections 296 - April 2nd, 2024

Testing GPT with RAG grouping...
  [4/20] done: NYT Connections 316 - April 22nd, 2024

Testing GPT with RAG grouping...
  [5/20] done: NYT Connections 264 - March 1st, 2024

Testing GPT with RAG grouping...
  [6/20] done: NYT Connections 300 - April 6th, 2024

Testing GPT with RAG grouping...
  [7/20] done: NYT Connections 213 - January 10th, 2024

Testing GPT with RAG grouping...
  [8/20] done: NYT Connections 155 - November 13th, 2023

Testing GPT with RAG grouping...
  [9/20] done: NYT Connections 185 - December 13th, 2023

Testing GPT with RAG grouping...
  [10/20] done: NYT Connections 227 - January 24th, 2024

Testing GPT with RAG grouping...
  [11/20] done: NYT Connections 215 - January 12th, 2024

Testing GPT with R

# Baseline Evaluation

In [62]:
## Show Results for Baseline (no hints):
scores_no_hints, summary_no_hints = evaluate_run(
    raw_replies_without_hints,
    all_actual_without_hints,
)

Predicted,Actual
"✓ GARDENING TOOLSHOE, PLOW, RAKE, SICKLE","FARM TOOLSHOE, PLOW, RAKE, SICKLE"
"✓ TRICKS OR DECEPTIONSPLOT, PLOY, RUSE, TRICK","SCHEMEPLOT, PLOY, RUSE, TRICK"
"✓ ACTS OF ENJOYMENTAMUSE, DELIGHT, PLEASE, TICKLE","MAKE HAPPYAMUSE, DELIGHT, PLEASE, TICKLE"
"✓ SOUNDSBANG, PLOP, SPLASH, THUD","ONOMATOPOEIABANG, PLOP, SPLASH, THUD"


Predicted,Actual
"✓ DEGREESMAJOR, MINOR, DEGREE, CONCENTRATION","EAT VORACIOUSLYDOWN, INHALE, SCARF, WOLF"
"✓ BROAD TERMSGENERAL, BROAD, SWEEPING, BLANKET","AREAS OF ACADEMIC FOCUSCONCENTRATION, DEGREE, MAJOR, MINOR"
"✗ WOLF TERMSWOLF, CADET, DOWN, SCARFbest match: EAT VORACIOUSLY (3/4)","UNIVERSALBLANKET, BROAD, GENERAL, SWEEPING"
"✗ HEATINGHEATER, BAR, INHALE, STATIONbest match: SPACE ___ (3/4)","SPACE ___BAR, CADET, HEATER, STATION"


Predicted,Actual
"✓ VERBS FOR PRODDINGJAB, POKE, PROD, STICK","THRUSTJAB, POKE, PROD, STICK"
"✓ TYPES OF SNAKESADDER, BOA, MAMBA, MOCCASIN","KINDS OF SNAKESADDER, BOA, MAMBA, MOCCASIN"
"✗ KIND OF SHOESSLIPPER, BOWTIE, BALL, TUBEbest match: SEEN IN “CINDERELLA” (2/4)","SEEN IN “CINDERELLA”BALL, PRINCE, PUMPKIN, SLIPPER"
"✗ PRINCE AND PUMPKINPRINCE, PUMPKIN, WHEEL, ELBOWbest match: SEEN IN “CINDERELLA” (2/4)","PASTA SHAPESBOWTIE, ELBOW, TUBE, WHEEL"


Predicted,Actual
"✓ SYNONYMS FOR FRIENDBUD, CHUM, MATE, PAL","SLANG FOR FRIENDBUD, CHUM, MATE, PAL"
"✓ WORDS DESCRIBING MOISTURE OR TEMPERATURECOLD, DANK, DARK, MUSTY","ADJECTIVES FOR A BASEMENTCOLD, DANK, DARK, MUSTY"
"✗ FELINE-RELATED TERMSKITTY, WHISKERS, FUZZ, SCRUFFbest match: STUBBLE (3/4)","STUBBLEFUZZ, SCRUFF, SHADOW, WHISKERS"
"✗ WORDS RELATED TO POOL OR CONTAINERFUND, POOL, POT, SHADOWbest match: COLLECTION OF MONEY (3/4)","COLLECTION OF MONEYFUND, KITTY, POOL, POT"


Predicted,Actual
"✓ FOOD INGREDIENTSBREAD, BUTTER, GARLIC, PARSLEY","GARLIC BREAD INGREDIENTSBREAD, BUTTER, GARLIC, PARSLEY"
"✓ GAMBLING TERMSBET, GAMBLE, RISK, STAKE","WAGERBET, GAMBLE, RISK, STAKE"
"✗ GENRES OF FICTIONADVENTURE, FANTASY, FRONTIER, VAMPIREbest match: DISNEYLAND LANDS (3/4)","DISNEYLAND LANDSADVENTURE, FANTASY, FRONTIER, TOMORROW"
"✗ SPORTSBASEBALL, CRICKET, FRUIT, TOMORROWbest match: ___ BAT (3/4)","___ BATBASEBALL, CRICKET, FRUIT, VAMPIRE"


...

Predicted,Actual
"✓ VERBS ASSOCIATED WITH MOVEMENTDODGE, DUCK, ESCAPE, SKIRT","AVOIDDODGE, DUCK, ESCAPE, SKIRT"
"✗ TYPES OF BIRDSGOOSE, ROBIN, HOBBES, BIRDSbest match: SIDEKICKS (3/4)","HITCHCOCK MOVIESBIRDS, NOTORIOUS, REBECCA, ROPE"
"✗ LITERARY WORKS OR CHARACTERSREBECCA, WATSON, NOTORIOUS, CREAMbest match: HITCHCOCK MOVIES (2/4)","SIDEKICKSGOOSE, HOBBES, ROBIN, WATSON"
"✗ STRING-RELATED TERMSROPE, SAY, STRING, COTTAGEbest match: ___ CHEESE (3/4)","___ CHEESECOTTAGE, CREAM, SAY, STRING"


Predicted,Actual
"✓ SYNONYMS FOR RELAXCHILL, LOAF, LOUNGE, HANG","RELAXCHILL, HANG, LOAF, LOUNGE"
"✓ MUSICAL TERMSBANGER, BOP, GROOVE, JAM","CATCHY SONGBANGER, BOP, GROOVE, JAM"
"✓ FOOD ITEMSSCONE, TRIFLE, MASH, ROAST","BRITISH CUISINEMASH, ROAST, SCONE, TRIFLE"
"✓ VERBS FOR MANIPULATIONBIND, PICKLE, SCRAPE, SPOT","STICKY SITUATIONBIND, PICKLE, SCRAPE, SPOT"


Predicted,Actual
"✓ CONJUNCTIONSHOWEVER, STILL, THOUGH, YET","NEVERTHELESSHOWEVER, STILL, THOUGH, YET"
"✗ VERBSHEAR, KNOCK, SEE, FLUSHbest match: REPEATED WORDS IN EXPRESSIONS (2/4)","REPEATED WORDS IN EXPRESSIONSHEAR, KNOCK, THERE, TUT"
"✗ PRONOUNSARE, YOU, WE, THEREbest match: WORDS ABBREVIATED WITH LETTERS (2/4)","WORDS ABBREVIATED WITH LETTERSARE, SEE, WHY, YOU"
"✗ FAMILY RELATEDTUT, JELLY, FAMILY, WHYbest match: ROYAL ___ (2/4)","ROYAL ___FAMILY, FLUSH, JELLY, WE"


Predicted,Actual
"✓ FLOWERSDAISY, ROSE, TULIP, VIOLET","FLOWERSDAISY, ROSE, TULIP, VIOLET"
"✓ FARMINGBARN, CHICKEN, FARMER, TRACTOR","SEEN ON A FARMBARN, CHICKEN, FARMER, TRACTOR"
"✗ ADJECTIVESCRAVEN, WAN, DUST, YELLOWbest match: HORROR DIRECTORS (2/4)","HORROR DIRECTORSASTER, CARPENTER, CRAVEN, WAN"
"✗ LIFE ACTIVITIESLIFE, SPORTS, ASTR, CARPENTERbest match: ___ JACKET (2/4)","___ JACKETDUST, LIFE, SPORTS, YELLOW"


Contest,Solved,Group Acc,Partial-3,Partial-2
"NYT Connections 188 - December 16th, 2023",✓,100%,100%,100%
"NYT Connections 272 - March 9th, 2024",✗,50%,100%,100%
"NYT Connections 296 - April 2nd, 2024",✗,50%,50%,100%
"NYT Connections 316 - April 22nd, 2024",✗,50%,100%,100%
"NYT Connections 264 - March 1st, 2024",✗,50%,100%,100%
"NYT Connections 300 - April 6th, 2024",✗,50%,100%,100%
"NYT Connections 213 - January 10th, 2024",✗,50%,100%,100%
"NYT Connections 155 - November 13th, 2023",✓,100%,100%,100%
"NYT Connections 185 - December 13th, 2023",✓,100%,100%,100%
"NYT Connections 227 - January 24th, 2024",✗,50%,100%,100%


In [23]:
# Wordnet Evaluation

In [63]:
## Show Results for WordNet:
scores_with_hints, summary_with_hints = evaluate_run(
    raw_replies_with_hints,
    all_actual_with_hints
)

Predicted,Actual
"✓ TOOLS FOR FARMINGHOE, PLOW, RAKE, SICKLE","FARM TOOLSHOE, PLOW, RAKE, SICKLE"
"✓ DECEPTIVE STRATEGIESPLOT, PLOY, RUSE, TRICK","SCHEMEPLOT, PLOY, RUSE, TRICK"
"✓ ACTS OF PLEASUREAMUSE, DELIGHT, PLEASE, TICKLE","MAKE HAPPYAMUSE, DELIGHT, PLEASE, TICKLE"
"✓ NOISES OF IMPACTBANG, PLOP, SPLASH, THUD","ONOMATOPOEIABANG, PLOP, SPLASH, THUD"


Predicted,Actual
"✓ THINGS TO WEARDOWN, SCARF, WOLF, INHALE","EAT VORACIOUSLYDOWN, INHALE, SCARF, WOLF"
"✓ AREAS OF STUDYCONCENTRATION, DEGREE, MAJOR, MINOR","AREAS OF ACADEMIC FOCUSCONCENTRATION, DEGREE, MAJOR, MINOR"
"✓ GENERAL TERMSBLANKET, BROAD, GENERAL, SWEEPING","UNIVERSALBLANKET, BROAD, GENERAL, SWEEPING"
"✓ PLACES OR POSITIONSBAR, CADET, HEATER, STATION","SPACE ___BAR, CADET, HEATER, STATION"


Predicted,Actual
"✓ GESTURESJAB, POKE, PROD, STICK","THRUSTJAB, POKE, PROD, STICK"
"✓ SNAKESADDER, BOA, MAMBA, MOCCASIN","KINDS OF SNAKESADDER, BOA, MAMBA, MOCCASIN"
"✓ FOOTWEAR AND ROYALTYBALL, PRINCE, PUMPKIN, SLIPPER","SEEN IN “CINDERELLA”BALL, PRINCE, PUMPKIN, SLIPPER"
"✓ OBJECTSBOWTIE, ELBOW, TUBE, WHEEL","PASTA SHAPESBOWTIE, ELBOW, TUBE, WHEEL"


Predicted,Actual
"✓ FRIENDSHIP TERMSBUD, CHUM, MATE, PAL","SLANG FOR FRIENDBUD, CHUM, MATE, PAL"
"✓ MUSTY ENVIRONMENTSCOLD, DANK, DARK, MUSTY","ADJECTIVES FOR A BASEMENTCOLD, DANK, DARK, MUSTY"
"✓ BODY PARTS/CHARACTERISTICSFUZZ, SCRUFF, SHADOW, WHISKERS","STUBBLEFUZZ, SCRUFF, SHADOW, WHISKERS"
"✓ MONEY POOL TERMSFUND, KITTY, POOL, POT","COLLECTION OF MONEYFUND, KITTY, POOL, POT"


Predicted,Actual
"✓ FOOD ITEMSBREAD, BUTTER, GARLIC, PARSLEY","GARLIC BREAD INGREDIENTSBREAD, BUTTER, GARLIC, PARSLEY"
"✓ GAMBLING TERMSBET, GAMBLE, RISK, STAKE","WAGERBET, GAMBLE, RISK, STAKE"
"✓ CONCEPTS OF TIME AND IMAGINATIONADVENTURE, FANTASY, FRONTIER, TOMORROW","DISNEYLAND LANDSADVENTURE, FANTASY, FRONTIER, TOMORROW"
"✓ GAMES AND MYTHICAL BEINGSBASEBALL, CRICKET, FRUIT, VAMPIRE","___ BATBASEBALL, CRICKET, FRUIT, VAMPIRE"


...

Predicted,Actual
"✓ ACTIONS TO AVOIDDODGE, DUCK, ESCAPE, SKIRT","AVOIDDODGE, DUCK, ESCAPE, SKIRT"
"✗ TYPES OF BIRDSGOOSE, DUCK, ROBIN, BIRDSbest match: SIDEKICKS (2/4)","HITCHCOCK MOVIESBIRDS, NOTORIOUS, REBECCA, ROPE"
"✗ CHARACTERS IN LITERATUREHOBBES, WATSON, REBECCA, NOTORIOUSbest match: HITCHCOCK MOVIES (2/4)","SIDEKICKSGOOSE, HOBBES, ROBIN, WATSON"
"✗ HOUSE AND ITS COMPONENTSCOTTAGE, CREAM, ROPE, STRINGbest match: ___ CHEESE (3/4)","___ CHEESECOTTAGE, CREAM, SAY, STRING"


Predicted,Actual
"✓ CASUAL ACTIVITIESCHILL, HANG, LOAF, LOUNGE","RELAXCHILL, HANG, LOAF, LOUNGE"
"✓ MUSIC TERMSBANGER, BOP, GROOVE, JAM","CATCHY SONGBANGER, BOP, GROOVE, JAM"
"✓ DESSERTSMASH, ROAST, SCONE, TRIFLE","BRITISH CUISINEMASH, ROAST, SCONE, TRIFLE"
"✓ FOOD PREPARATIONBIND, PICKLE, SCRAPE, SPOT","STICKY SITUATIONBIND, PICKLE, SCRAPE, SPOT"


Predicted,Actual
"✓ CONJUNCTIONSHOWEVER, STILL, THOUGH, YET","NEVERTHELESSHOWEVER, STILL, THOUGH, YET"
"✓ SOUNDS / NOISESHEAR, KNOCK, THERE, TUT","REPEATED WORDS IN EXPRESSIONSHEAR, KNOCK, THERE, TUT"
"✓ PRONOUNS / QUESTIONSARE, SEE, WHY, YOU","WORDS ABBREVIATED WITH LETTERSARE, SEE, WHY, YOU"
"✓ FAMILY / RELATIONSHIPSFAMILY, FLUSH, JELLY, WE","ROYAL ___FAMILY, FLUSH, JELLY, WE"


Predicted,Actual
"✓ FLOWERSDAISY, ROSE, TULIP, VIOLET","FLOWERSDAISY, ROSE, TULIP, VIOLET"
"✓ FARMINGBARN, CHICKEN, FARMER, TRACTOR","SEEN ON A FARMBARN, CHICKEN, FARMER, TRACTOR"
"✗ CHARACTERISTICSCARPENTER, CRAVEN, WAN, DUSTbest match: HORROR DIRECTORS (3/4)","HORROR DIRECTORSASTER, CARPENTER, CRAVEN, WAN"
"✗ LIFE ASPECTSLIFE, SPORTS, YELLOWbest match: ___ JACKET (3/4)","___ JACKETDUST, LIFE, SPORTS, YELLOW"


Contest,Solved,Group Acc,Partial-3,Partial-2
"NYT Connections 188 - December 16th, 2023",✓,100%,100%,100%
"NYT Connections 272 - March 9th, 2024",✓,100%,100%,100%
"NYT Connections 296 - April 2nd, 2024",✓,100%,100%,100%
"NYT Connections 316 - April 22nd, 2024",✓,100%,100%,100%
"NYT Connections 264 - March 1st, 2024",✓,100%,100%,100%
"NYT Connections 300 - April 6th, 2024",✓,100%,100%,100%
"NYT Connections 213 - January 10th, 2024",✓,100%,100%,100%
"NYT Connections 155 - November 13th, 2023",✓,100%,100%,100%
"NYT Connections 185 - December 13th, 2023",✓,100%,100%,100%
"NYT Connections 227 - January 24th, 2024",✓,100%,100%,100%


# RAG Evaluation

In [64]:
## Show Results for WordNet:
scores_with_RAG, summary_with_RAG = evaluate_run(
    raw_replies_with_RAG,
    all_actual_with_RAG
)

Predicted,Actual
"✓ FARMING TOOLSHOE, PLOW, RAKE, SICKLE","FARM TOOLSHOE, PLOW, RAKE, SICKLE"
"✓ DECEPTIONPLOT, PLOY, RUSE, TRICK","SCHEMEPLOT, PLOY, RUSE, TRICK"
"✓ ENTERTAINMENTAMUSE, DELIGHT, PLEASE, TICKLE","MAKE HAPPYAMUSE, DELIGHT, PLEASE, TICKLE"
"✓ SOUNDSBANG, PLOP, SPLASH, THUD","ONOMATOPOEIABANG, PLOP, SPLASH, THUD"


Predicted,Actual
"✗ COLLEGE TERMSMAJOR, MINOR, DEGREE, CADETbest match: AREAS OF ACADEMIC FOCUS (3/4)","EAT VORACIOUSLYDOWN, INHALE, SCARF, WOLF"
"✗ HEATING OBJECTSHEATER, BLANKET, SCARF, DOWNbest match: EAT VORACIOUSLY (2/4)","AREAS OF ACADEMIC FOCUSCONCENTRATION, DEGREE, MAJOR, MINOR"
"✗ WIDE OR GENERALBROAD, GENERAL, SWEEPING, CONCENTRATIONbest match: UNIVERSAL (3/4)","UNIVERSALBLANKET, BROAD, GENERAL, SWEEPING"
"✗ ANIMALS / ACTIONSWOLF, INHALE, BAR, STATIONbest match: EAT VORACIOUSLY (2/4)","SPACE ___BAR, CADET, HEATER, STATION"


Predicted,Actual
"✓ SNAKESADDER, BOA, MAMBA, MOCCASIN","THRUSTJAB, POKE, PROD, STICK"
"✓ PUSHING ACTIONSJAB, POKE, PROD, STICK","KINDS OF SNAKESADDER, BOA, MAMBA, MOCCASIN"
"✗ FOOTWEARSLIPPER, BOWTIE, TUBE, WHEELbest match: PASTA SHAPES (3/4)","SEEN IN “CINDERELLA”BALL, PRINCE, PUMPKIN, SLIPPER"
"✗ VAPOROUS VEGETABLEBALL, PRINCE, PUMPKIN, WHEELbest match: SEEN IN “CINDERELLA” (3/4)","PASTA SHAPESBOWTIE, ELBOW, TUBE, WHEEL"


Predicted,Actual
"✓ FRIENDSBUD, CHUM, MATE, PAL","SLANG FOR FRIENDBUD, CHUM, MATE, PAL"
"✓ UNPLEASANT ODORSCOLD, DANK, DARK, MUSTY","ADJECTIVES FOR A BASEMENTCOLD, DANK, DARK, MUSTY"
"✗ PET NAMESKITTY, WHISKERS, FUZZ, SCRUFFbest match: STUBBLE (3/4)","STUBBLEFUZZ, SCRUFF, SHADOW, WHISKERS"
"✗ GAMBLING TERMSPOOL, POT, FUND, BANKbest match: COLLECTION OF MONEY (3/4)","COLLECTION OF MONEYFUND, KITTY, POOL, POT"


Predicted,Actual
"✓ FOODSBREAD, BUTTER, GARLIC, PARSLEY","GARLIC BREAD INGREDIENTSBREAD, BUTTER, GARLIC, PARSLEY"
"✓ RISKY ACTIVITIESBET, GAMBLE, RISK, STAKE","WAGERBET, GAMBLE, RISK, STAKE"
"✓ ADVENTURE GENRESADVENTURE, FANTASY, FRONTIER, TOMORROW","DISNEYLAND LANDSADVENTURE, FANTASY, FRONTIER, TOMORROW"
"✓ SPORTSBASEBALL, CRICKET, FRUIT, VAMPIRE","___ BATBASEBALL, CRICKET, FRUIT, VAMPIRE"


...

Predicted,Actual
"✓ VERBS MEANING TO AVOIDDODGE, DUCK, ESCAPE, SKIRT","AVOIDDODGE, DUCK, ESCAPE, SKIRT"
"✗ BIRD NAMESGOOSE, ROBIN, BIRDS, HOBBESbest match: SIDEKICKS (3/4)","HITCHCOCK MOVIESBIRDS, NOTORIOUS, REBECCA, ROPE"
"✗ NOTORIOUS FICTIONAL CHARACTERSNOTORIOUS, REBECCA, WATSON, ROPEbest match: HITCHCOCK MOVIES (3/4)","SIDEKICKSGOOSE, HOBBES, ROBIN, WATSON"
"✓ FOOD RELATED TERMSCREAM, COTTAGE, SAY, STRING","___ CHEESECOTTAGE, CREAM, SAY, STRING"


Predicted,Actual
"✓ ACTIVITIES FOR RELAXATIONCHILL, LOAF, LOUNGE, HANG","RELAXCHILL, HANG, LOAF, LOUNGE"
"✓ MUSIC TERMSBOP, GROOVE, JAM, BANGER","CATCHY SONGBANGER, BOP, GROOVE, JAM"
"✓ FOOD ITEMSMASH, ROAST, SCONE, TRIFLE","BRITISH CUISINEMASH, ROAST, SCONE, TRIFLE"
"✓ ACTIONSBIND, PICKLE, SCRAPE, SPOT","STICKY SITUATIONBIND, PICKLE, SCRAPE, SPOT"


Predicted,Actual
"✓ CONJUNCTIONSHOWEVER, STILL, THOUGH, YET","NEVERTHELESSHOWEVER, STILL, THOUGH, YET"
"✗ VERBS RELATED TO PERCEPTION OR COMMUNICATIONHEAR, SEE, KNOCK, TUTbest match: REPEATED WORDS IN EXPRESSIONS (3/4)","REPEATED WORDS IN EXPRESSIONSHEAR, KNOCK, THERE, TUT"
"✗ STATEMENTS OF EXISTENCEARE, THERE, YOU, WEbest match: WORDS ABBREVIATED WITH LETTERS (2/4)","WORDS ABBREVIATED WITH LETTERSARE, SEE, WHY, YOU"
"✗ FOOD ITEMSFAMILY, FLUSH, JELLY, YOUbest match: ROYAL ___ (3/4)","ROYAL ___FAMILY, FLUSH, JELLY, WE"


Predicted,Actual
"✓ FLOWERSDAISY, ROSE, TULIP, VIOLET","FLOWERSDAISY, ROSE, TULIP, VIOLET"
"✓ FARMINGBARN, CHICKEN, FARMER, TRACTOR","SEEN ON A FARMBARN, CHICKEN, FARMER, TRACTOR"
"✗ COLOR DESCRIPTORSASTRER, WAN, YELLOW, DUSTbest match: ___ JACKET (2/4)","HORROR DIRECTORSASTER, CARPENTER, CRAVEN, WAN"
"✗ CHARACTER TRAITSCRAVEN, LIFE, SPORTS, CHICKENbest match: ___ JACKET (2/4)","___ JACKETDUST, LIFE, SPORTS, YELLOW"


Contest,Solved,Group Acc,Partial-3,Partial-2
"NYT Connections 188 - December 16th, 2023",✓,100%,100%,100%
"NYT Connections 272 - March 9th, 2024",✗,0%,50%,100%
"NYT Connections 296 - April 2nd, 2024",✗,50%,100%,100%
"NYT Connections 316 - April 22nd, 2024",✗,50%,100%,100%
"NYT Connections 264 - March 1st, 2024",✓,100%,100%,100%
"NYT Connections 300 - April 6th, 2024",✗,50%,100%,100%
"NYT Connections 213 - January 10th, 2024",✗,50%,100%,100%
"NYT Connections 155 - November 13th, 2023",✓,100%,100%,100%
"NYT Connections 185 - December 13th, 2023",✓,100%,100%,100%
"NYT Connections 227 - January 24th, 2024",✗,50%,100%,100%


In [65]:
## Comparison: Baseline vs WordNet vs RAG
from IPython.display import display, HTML

metrics = ["gcr", "acc", "p3", "p2"]
labels  = ["Game Completion Rate", "Group Accuracy", "Partial-3 Rate", "Partial-2 Rate"]

def _delta_cell(a, b):
    diff = b - a
    color = "#66bb6a" if diff > 0 else ("#ef5350" if diff < 0 else "#aaa")
    arrow = "▲" if diff > 0 else ("▼" if diff < 0 else "—")
    return (f'<td style="padding:6px 14px;text-align:center;color:{color};font-weight:bold">'
            f'{arrow} {abs(diff):.1%}</td>')

header = (
    '<tr style="color:#aaa;border-bottom:1px solid #555">'
    '<th style="padding:6px 14px;text-align:left">Metric</th>'
    '<th style="padding:6px 14px">Baseline (A)</th>'
    '<th style="padding:6px 14px">WordNet (B)</th>'
    '<th style="padding:6px 14px">RAG (C)</th>'
    '<th style="padding:6px 14px">Δ WordNet (B − A)</th>'
    '<th style="padding:6px 14px">Δ RAG (C − A)</th>'
    '</tr>'
)
rows = "".join(
    f'<tr>'
    f'<td style="padding:6px 14px">{lbl}</td>'
    f'<td style="padding:6px 14px;text-align:center">{summary_no_hints[m]:.1%}</td>'
    f'<td style="padding:6px 14px;text-align:center">{summary_with_hints[m]:.1%}</td>'
    f'<td style="padding:6px 14px;text-align:center">{summary_with_RAG[m]:.1%}</td>'
    f'{_delta_cell(summary_no_hints[m], summary_with_hints[m])}'
    f'{_delta_cell(summary_no_hints[m], summary_with_RAG[m])}'
    f'</tr>'
    for m, lbl in zip(metrics, labels)
)

display(HTML(
    '<h2 style="font-family:monospace;border-bottom:2px solid #555;padding-bottom:6px">'
    'Comparison: Baseline vs WordNet vs RAG</h2>'
    '<table style="border-collapse:collapse;font-family:monospace;font-size:0.95em">'
    f'{header}{rows}</table>'
    '<p style="color:#888;font-size:0.85em;font-family:monospace">'
    '▲ = Better than Baseline &nbsp;|&nbsp; ▼ = Worse than Baseline</p>'
))


Metric,Baseline (A),WordNet (B),RAG (C),Δ WordNet (B − A),Δ RAG (C − A)
Game Completion Rate,35.0%,90.0%,35.0%,▲ 55.0%,— 0.0%
Group Accuracy,61.0%,93.8%,61.3%,▲ 32.8%,▲ 0.3%
Partial-3 Rate,83.5%,97.5%,86.2%,▲ 14.0%,▲ 2.8%
Partial-2 Rate,100.0%,100.0%,100.0%,— 0.0%,— 0.0%


In [79]:
# Matching Game name with diffculty level
puzzle_difficulty = {}
for p in pool:
    puzzle_difficulty[p["contest"]] = p.get("difficulty")

# Print header
print(f"{'Band':<8}  {'Baseline':>10}  {'WordNet':>10}  {'RAG':>10}")

# Go through each difficulty band
for lo, hi in [(1, 2), (2, 3), (3, 4), (4, 5)]:

    # Baseline
    b_total = 0
    b_count = 0
    for s in scores_no_hints:
        d = puzzle_difficulty.get(s["contest"])
        if d is not None and lo <= d < hi:
            b_total += s["group_acc"]
            b_count += 1
    b_avg = f"{b_total / b_count:.1%}" if b_count > 0 else "N/A"

    #Database
    w_total = 0
    w_count = 0
    for s in scores_with_hints:
        d = puzzle_difficulty.get(s["contest"])
        if d is not None and lo <= d < hi:
            w_total += s["group_acc"]
            w_count += 1
    w_avg = f"{w_total / w_count:.1%}" if w_count > 0 else "N/A"

    #RAG
    r_total = 0
    r_count = 0
    for s in scores_with_RAG:
        d = puzzle_difficulty.get(s["contest"])
        if d is not None and lo <= d < hi:
            r_total += s["group_acc"]
            r_count += 1
    r_avg = f"{r_total / r_count:.1%}" if r_count > 0 else "N/A"

    print(f"{lo}-{hi:<6}  {b_avg:>10}  {w_avg:>10}  {r_avg:>10}")


Band        Baseline     WordNet         RAG
1-2            60.0%      100.0%       60.0%
2-3            70.0%      100.0%       70.0%
3-4            74.0%      100.0%       70.0%
4-5            40.0%       75.0%       45.0%
